# Reproduction Manual: High-Amplitude Frame Removal and Orthogonally Constrained DeepSets for Autism Classification

Notation: `RSS(t)` co-fluctuation amplitude; `R` FC matrix; `M` edge-weight matrix; `d` standardized effect size; `h = 32`.


## 1. Quality Control: Flat Time Series and Run Selection


In [ ]:
import numpy as np

FLAT_TS_TOL = 1e-6


def pick_first_valid_run(ts, segs, tol=FLAT_TS_TOL):
    """First non-flat run; (None, 0) if every run is flat."""
    if len(segs) < 2 or sum(segs) != ts.shape[0]:
        return ts, 1
    pos = 0
    for i, L in enumerate(segs, start=1):
        seg = ts[pos:pos + L]
        if float(np.abs(seg).max()) >= tol:
            return seg, i
        pos += L
    return None, 0


def load_and_screen(ts_np, segs, tol=FLAT_TS_TOL):
    """Usable -> (tensor, None); otherwise (None, "nan" | "flat")."""
    import torch
    seg, _ = pick_first_valid_run(ts_np, segs)
    if seg is not None:
        ts_np = np.ascontiguousarray(seg)
    ts = torch.from_numpy(ts_np)
    if not bool(torch.isfinite(ts).all()):
        return None, "nan"
    if float(ts.abs().max()) < FLAT_TS_TOL:
        return None, "flat"
    return ts, None


## 2. Frame-Wise Co-Fluctuation Amplitude

$$\mathrm{RSS}(t)=\sqrt{\sum_{i<j} e_{ij}(t)^2}=\sqrt{\tfrac{1}{2}\Big[\big(\textstyle\sum_i x_i^2\big)^2-\sum_i x_i^4\Big]}$$


In [ ]:
def amp_from_z(z):
    """RSS(t) = sqrt(1/2 [(sum_i z_i^2)^2 - sum_i z_i^4]); z: (T, K)."""
    z2 = z * z
    quad = 0.5 * (z2.sum(1) ** 2 - (z2 * z2).sum(1))
    return np.sqrt(np.maximum(quad, 0.0))


![](fig1_frame_subject.png)


## 3. Retained-Frame Mask

$$\mathcal{K}=\{\,t:\mathrm{RSS}(t)\le q_{0.93}\,\},\qquad R^{\mathcal{K}}=\mathrm{corr}\big(\bm{x}^{\mathcal{K}}\big)$$


In [ ]:
def _nframes(q, T):
    """Frames to discard: 0 if q <= 0, otherwise at least 1."""
    return 0 if q <= 0 else max(1, int(round(q * T)))


def frame_keep_mask(amp, spec):
    """Kept-frame mask (T,) for "low:q" / "high:q" / "both:qh,ql"."""
    T = amp.shape[0]
    side, _, qstr = spec.partition(":")
    qv = [float(x) for x in qstr.split(",")]
    order = np.argsort(amp.astype(np.float64))
    if side == "low":
        drop = order[:_nframes(qv[0], T)]
    elif side == "high":
        drop = order[T - _nframes(qv[0], T):]
    elif side == "both":
        kh, kl = _nframes(qv[0], T), _nframes(qv[1], T)
        drop = np.r_[order[T - kh:], order[:kl]]
    else:
        raise ValueError(spec)
    m = np.ones(T, bool)
    m[np.asarray(drop, dtype=np.int64)] = False
    return m


## 4. DeepSets Model and Row-Orthogonality Constraint

$$z=\mathbf{a}^{\top}\Big(\tfrac{1}{T}\sum_t g(W_1\mathbf{x}_t)\Big)+b,\qquad \mathbf{a}=\mathbf{w}_1-\mathbf{w}_0,\qquad g(u)=u\,\sigma(u)$$

$$W_1\leftarrow\big(W_1W_1^{\top}+\epsilon I_h\big)^{-1/2}W_1,\qquad \epsilon=10^{-6}$$


In [ ]:
import torch
import torch.nn as nn

ACT = "silu"


def gate_torch(u):
    """SiLU gate."""
    return torch.nn.functional.silu(u)


def gate_np(u):
    """SiLU gate, numpy."""
    return u / (1.0 + np.exp(-u))


class FrameGate(nn.Module):
    """SiLU gate for nn.Sequential."""

    def forward(self, z):
        return gate_torch(z)


def stiefel_retract(W):
    """Polar retraction onto {W: WW^T = I_h}."""
    h = W.shape[0]
    G_ = W @ W.t() + 1e-6 * torch.eye(h, device=W.device, dtype=W.dtype)
    S, U = torch.linalg.eigh(G_)
    inv_sqrt = U * (1.0 / S.clamp_min(1e-12).sqrt()) @ U.t()
    return inv_sqrt @ W


class BOLDAutoencoder(nn.Module):
    """DeepSets: shared frame MLP -> masked mean over frames -> linear head."""

    DEFAULT_HIDDEN = 32

    def __init__(self, n_rois: int = 120, hidden: int = DEFAULT_HIDDEN):
        super().__init__()
        self.phi = nn.Sequential(nn.Linear(n_rois, hidden, bias=False), FrameGate())
        self.head = nn.Linear(hidden, 2)
        nn.init.orthogonal_(self.phi[0].weight)

    def _masked_pool(self, h, mask):
        w = mask.unsqueeze(-1)
        return (h * w).sum(1) / mask.sum(1, keepdim=True).clamp(min=1e-8)

    def _encode(self, x, mask):
        return self.head(self._masked_pool(self.phi(x), mask))

    def forward(self, x, mask):
        """x: (B, T, K), mask: (B, T) -> logits (B, 2)."""
        return self._encode(x, mask)


## 5. Consequence for the Edge-Weight Matrix

$$\mathrm{spec}_{\neq0}(M)=\{a_i\}_{i=1}^{h},\qquad \|M\|_F=\|\mathbf{a}\|_2,\qquad \mathrm{rank}(M)\le h,\qquad M=W_1^{\top}\mathrm{diag}(\mathbf{a})W_1$$


In [ ]:
def spectrum_check(W1, a):
    """Non-zero spectrum of M versus the head weights a; also checks WW^T = I_h."""
    W1 = np.asarray(W1, float)
    a = np.asarray(a, float)
    M = W1.T @ (a[:, None] * W1)
    lam = np.sort(np.abs(np.linalg.eigvalsh(M)))[::-1][:len(a)]
    aa = np.sort(np.abs(a))[::-1]
    return dict(max_abs_dev=float(np.abs(lam - aa).max()),
                max_orth_dev=float(np.abs(W1 @ W1.T - np.eye(W1.shape[0])).max()),
                rank=int(np.linalg.matrix_rank(M)),
                fro_vs_a=float(abs(np.linalg.norm(M) - np.linalg.norm(a))))


## 6. Geometry of the Learned Read-Out

$$\mathrm{PR}=\frac{(\sum_i\lambda_i^2)^2}{\sum_i\lambda_i^4},\qquad \lambda_i\in\mathrm{spec}_{\neq0}(M),\qquad \mathrm{PR}\le h$$


In [ ]:
def readout_geometry(W1_list, a_list):
    """Filter norm, edge-weight energy and participation ratio -> (mean, SD)."""
    wn, fr, pr = [], [], []
    for W1, a in zip(W1_list, a_list):
        W1 = np.asarray(W1, float)
        a = np.asarray(a, float)
        wn.append(float(np.linalg.norm(W1, axis=1).mean()))
        M = W1.T @ (a[:, None] * W1)
        fr.append(float(np.linalg.norm(M)))
        ev = np.linalg.eigvalsh(M)
        ev = ev[np.abs(ev) > 1e-9 * max(1.0, float(np.abs(ev).max()))]
        pr.append(float((ev ** 2).sum() ** 2 / (ev ** 4).sum()))
    f = lambda v: (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0)
    return dict(wnorm=f(wn), fro=f(fr), pr=f(pr))


## 7. Aggregate Standardized Effect Size

$$d=\frac{\sqrt{\frac{1}{m}\sum_k\big(\bar r_k^{\mathrm{ASD}}-\bar r_k^{\mathrm{HC}}\big)^2}}{\sqrt{\frac{1}{m}\sum_k\big(s_k^{\mathrm{pooled}}\big)^2}}$$


In [ ]:
def effect_stats(s, n):
    """d and its two norms from per-edge moments.

    s = [sum_ASD r, sum_ASD r^2, sum_HC r, sum_HC r^2] (K x K, device); n = (n_ASD, n_HC).
    """
    n0, n1 = n
    summ = [x.double().cpu().numpy() for x in s]
    m0, m1 = summ[0] / n0, summ[2] / n1
    v0 = np.maximum((summ[1] - n0 * m0 * m0) / (n0 - 1), 0.0)
    v1 = np.maximum((summ[3] - n1 * m1 * m1) / (n1 - 1), 0.0)
    sd = np.sqrt(((n0 - 1) * v0 + (n1 - 1) * v1) / (n0 + n1 - 2))
    rms_d = float(np.sqrt(np.mean((m0 - m1) ** 2)))
    rms_sd = float(np.sqrt(np.mean(sd ** 2)))
    return rms_d / max(rms_sd, 1e-12), rms_d, rms_sd


## 8. Within-Group Homogeneity

$$\bar\rho=\frac{\big\|\sum_i\tilde{\mathbf f}_i\big\|^2-\sum_i\big\|\tilde{\mathbf f}_i\big\|^2}{n(n-1)m}$$


In [ ]:
def gram(z, keep):
    """Retained-frame re-standardized edge-wise r: x~^T x~ / T; z: (T, K) device float32."""
    x = z[keep]
    x = x - x.mean(0, keepdim=True)
    x = x / x.std(0, unbiased=False, keepdim=True).clamp_min(1e-8)
    return (x.t() @ x) / x.shape[0]


def rho_w(S, Q, n, m):
    """[||sum_i f~_i||^2 - sum_i ||f~_i||^2] / [n(n-1)m]."""
    return (float(S.double() @ S.double()) - Q) / (n * (n - 1) * m)


![](fig2_frame_removal.png)

![](fig3_edge_dose.png)


## 9. Linear Read-Out

$$g_{\text{odd}}(u)=\tfrac{u}{2},\qquad
g_{\text{even}}(u)=\tfrac{u}{2}\tanh\tfrac{u}{2}=\frac{u^2}{4}-\frac{u^4}{48}+\frac{u^6}{480}-\cdots$$

$$\langle M,R\rangle=\frac{T}{T-1}\sum_{i=1}^{h}a_i\,\overline{\big(\bm w_i^{\top}\bm x_t\big)^2},\qquad M=W_1^{\top}\mathrm{diag}(\mathbf a)W_1$$


In [ ]:
def fold_surrogates(ids, seqs, W1, a, b):
    """-> (z_quad, z_h, z_odd, z_even, H, M2); z_quad = <M, R>."""
    zq, zh, zo, ze, H, M2 = [], [], [], [], [], []
    for sid in ids:
        X = seqs[sid].numpy().astype(np.float64)
        T = X.shape[0]
        u = X @ W1.T
        m2 = (u ** 2).mean(0)
        g, g_neg = gate_np(u), gate_np(-u)
        h = g.mean(0)
        zq.append(T / (T - 1) * (a @ m2))
        zh.append(a @ h + b)
        ze.append(a @ ((g + g_neg) / 2).mean(0))
        zo.append(a @ ((g - g_neg) / 2).mean(0))
        H.append(h)
        M2.append(m2)
    return (np.array(zq), np.array(zh), np.array(zo), np.array(ze),
            np.array(H), np.array(M2))


def quad_scores(ids, seqs, W1, a):
    """<M, R> = T/(T-1) * sum_i a_i * m2_i, without materialising R."""
    out = []
    for sid in ids:
        X = seqs[sid].numpy().astype(np.float64)
        T = X.shape[0]
        m2 = ((X @ W1.T) ** 2).mean(0)
        out.append(T / (T - 1) * (a @ m2))
    return np.array(out)


def numeric_identity(X, W1, a):
    """<M, R> with an explicit R versus T/(T-1) * sum_i a_i * m2_i."""
    T = X.shape[0]
    M = W1.T @ (a[:, None] * W1)
    R = X.T @ X / (T - 1)
    lhs = float(np.sum(M * R))
    rhs = float(T / (T - 1) * (a @ ((X @ W1.T) ** 2).mean(0)))
    return lhs, rhs, abs(lhs - rhs)


## 10. Network Block Structure


In [ ]:
def block_mean(M, blocks):
    """Signed block means; NaN cells (ROI diagonal) are skipped."""
    out = np.full((len(blocks), len(blocks)), np.nan)
    for i, (si, ei) in enumerate(blocks):
        for j, (sj, ej) in enumerate(blocks):
            v = M[si:ei, sj:ej]
            v = v[np.isfinite(v)]
            if v.size:
                out[i, j] = v.mean()
    return out


def block_matrix(Mc, perm, blocks):
    """Reorder ROIs by network, drop the ROI diagonal, average within blocks."""
    Mr = Mc[np.ix_(perm, perm)].copy()
    np.fill_diagonal(Mr, np.nan)
    return block_mean(Mr, blocks)


def block_consistency(Ba, Bb):
    """Within-network (diagonal) and between-network agreement of two block matrices."""
    from scipy import stats
    n = Ba.shape[0]
    od = ~np.eye(n, dtype=bool)
    return dict(diag_r=float(np.corrcoef(np.diag(Ba), np.diag(Bb))[0, 1]),
                offdiag_r=float(np.corrcoef(Ba[od], Bb[od])[0, 1]),
                offdiag_spearman=float(stats.spearmanr(Ba[od], Bb[od]).statistic),
                n_offdiag_blocks=int(od.sum()))


![](fig5_block_matrices.png)


## 11. Sensitivity to the Removed Fraction

![](fig4_dose_response.png)


## 12. Similarity Between Repeated Scans of One Subject

![](fig6_multirun_fc.png)


## 13. Ten-Fold Cross-Validation Split


In [ ]:
from sklearn.model_selection import StratifiedKFold

SEED = 42


def make_folds(sub_ids, labels, kfold=10, seed=SEED):
    """Subject-level stratified k-fold, generated once per cohort."""
    skf = StratifiedKFold(n_splits=kfold, shuffle=True, random_state=seed)
    return [(f"fold{i}", sub_ids[tr], sub_ids[te])
            for i, (tr, te) in enumerate(skf.split(sub_ids, labels))]


## 14. Motion-Mask Control


In [ ]:
def fd_score(thrs, rem):
    """Per-frame score monotone in FD: the largest threshold still flagging the frame."""
    idx = np.where(rem, np.arange(rem.shape[0])[:, None], -1).max(0)
    return np.where(idx >= 0, thrs[idx], -1.0)


def motion_agreement(dropped, fds, q):
    """Overlap with the q fraction of frames with the largest FD (chance level = q)."""
    k = max(int(round(q * len(fds))), 1)
    order = np.argsort(-fds, kind="stable")
    mot_top = np.zeros(len(fds), bool)
    mot_top[order[:k]] = True
    inter = int((dropped & mot_top).sum())
    return inter / dropped.sum() if dropped.any() else float("nan")


## 15. Linear Surrogate Fitting


In [ ]:
def aligned_acc(z_src_tr, z_src_te, z_ref_tr, y_te):
    """Least-squares affine alignment of the surrogate to the model scale, thresholded at zero."""
    A = np.vstack([z_src_tr, np.ones_like(z_src_tr)]).T
    coef, *_ = np.linalg.lstsq(A, z_ref_tr, rcond=None)
    return float((((z_src_te * coef[0] + coef[1]) > 0) == (y_te == 1)).mean())


## 16. Paired Tests


In [ ]:
from scipy.stats import wilcoxon


def paired_p(per_sub_q):
    """Two-sided signed-rank test over the pooled subjects for one q."""
    return float(wilcoxon([np.mean(v) for v in per_sub_q.values()]).pvalue)


## 17. Cross-Cohort Protocol


In [ ]:
def exclude_overlap(sub_ids, labels, ex):
    """Symmetric removal of the subjects shared by both cohorts."""
    if not ex:
        return sub_ids, labels, 0
    keep = [i for i, s in enumerate(sub_ids) if int(s) not in ex]
    dropped = len(sub_ids) - len(keep)
    return sub_ids[keep], labels[keep], dropped
